In [1]:
pip install biopython primer3-py pandas


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 52.0 MB/s eta 0:00:00


In [2]:
#WINDOWS CREATION

from Bio import SeqIO #Bio.AlignIO is a module in Biopython used for reading and writing MSA files

input_fasta = "consensus.fa"  #This is the consensus file of aligned COVID-19 file
window_size = 200
step = 80

record = SeqIO.read(input_fasta, "fasta")

windows = []
for start in range(0, len(record.seq) - window_size + 1, step):
    end = start + window_size
    window_seq = record.seq[start:end]

    windows.append({
        "id": f"window_{start+1}_{end}",
        "seq": window_seq
    })

# Write windows to FASTA
with open("windows.fa", "w") as out:
    for w in windows:
        out.write(f">{w['id']}\n{w['seq']}\n")

print(f"Generated {len(windows)} windows")


Generated 372 windows


In [3]:
from Bio import AlignIO

alignment_file = "SARS.fa"
min_len = 17
allow_gaps = False   # set True ONLY if you really want gap-tolerant regions

alignment = AlignIO.read(alignment_file, "fasta")
seqs = [str(rec.seq) for rec in alignment]

num_seqs = len(seqs)
aln_len = len(seqs[0])

print(f"Sequences: {num_seqs}")
print(f"Alignment length: {aln_len}")

# ----------------------------------------
# STEP 1: Find conserved positions STRICTLY
# ----------------------------------------
strict_conserved = []

for i in range(aln_len):
    bases = {seq[i] for seq in seqs}

    if not allow_gaps and "-" in bases:
        strict_conserved.append(False)
    elif len(bases) == 1:
        strict_conserved.append(True)
    else:
        strict_conserved.append(False)

# ----------------------------------------
# STEP 2: Merge continuous conserved blocks
# ----------------------------------------
regions = []
start = None

for i, is_cons in enumerate(strict_conserved):
    if is_cons and start is None:
        start = i
    elif not is_cons and start is not None:
        end = i - 1
        if (end - start + 1) >= min_len:
            regions.append((start, end))
        start = None

# catch final region
if start is not None:
    end = aln_len - 1
    if (end - start + 1) >= min_len:
        regions.append((start, end))

# ----------------------------------------
# OUTPUT
# ----------------------------------------
print("\nHighly conserved continuous regions (≥17 bp):\n")

for idx, (start, end) in enumerate(regions, 1):
    seq = seqs[0][start:end+1]
    print(f"Region {idx}: {start+1}-{end+1} ({end-start+1} bp)")
    print(f"Sequence: {seq}\n")

Sequences: 107
Alignment length: 30410

Highly conserved continuous regions (≥17 bp):

Region 1: 1521-1537 (17 bp)
Sequence: TTTGGAGGCTGTGTGTT

Region 2: 3083-3099 (17 bp)
Sequence: TACCCTCCAGATGAGGA

Region 3: 3400-3426 (27 bp)
Sequence: TGGTTATTTAAAACTTACTGACAATGT

Region 4: 4041-4057 (17 bp)
Sequence: CTGGAAGAAACTAAGTT

Region 5: 7888-7907 (20 bp)
Sequence: TGTACAACTATTGTTAATGG

Region 6: 8014-8030 (17 bp)
Sequence: ATTAGTGATGAAGTTGC

Region 7: 8047-8063 (17 bp)
Sequence: CAGTTTAAAAGACCAAT

Region 8: 8730-8746 (17 bp)
Sequence: GTAGCAAAAAGTCACAA

Region 9: 9066-9083 (18 bp)
Sequence: TGGTGTCACTCGTGACAT

Region 10: 9131-9153 (23 bp)
Sequence: CATGGTTTAGCCAGCGTGGTGGT

Region 11: 10651-10668 (18 bp)
Sequence: CATTCTATGCAAAATTGT

Region 12: 10765-10790 (26 bp)
Sequence: TACAATGGTTCACCATCTGGTGTTTA

Region 13: 10843-10868 (26 bp)
Sequence: TCATGTGGTAGTGTTGGTTTTAACAT

Region 14: 11968-11989 (22 bp)
Sequence: GTTTATTGTTTCTTAGGCTATT

Region 15: 12043-12059 (17 bp)
Sequence: ACTCTTGGTGTTTATGA

#Prototype programme to exclude known primers, probes and continuous conserved stretches from other coronaviruses.

In [4]:
#Prototype programme to exclude known primers, probes and continuous conserved stretches from other coronaviruses.

from Bio import SeqIO

# INPUT FILES
windows_fasta = "windows.fa"     # candidate windows
exclude_file = "List_ExcludeExistingAssaysAndConservedContinuousStretches.txt"     # existng primers & other-CoV conserved regions

output_fasta = "filtered_windows.fa"

# LOAD EXCLUSION SEQUENCES
exclude_seqs = set()

with open(exclude_file) as f:
    for line in f:
        seq = line.strip().upper()
        if seq:
            exclude_seqs.add(seq)

print(f"Loaded {len(exclude_seqs)} exclusion sequences")

# FILTER WINDOWS
kept = []
removed = []

for record in SeqIO.parse(windows_fasta, "fasta"):
    window_seq = str(record.seq).upper()

    exclude_hit = False
    for ex in exclude_seqs:
        if ex in window_seq:
            exclude_hit = True
            break

    if exclude_hit:
        removed.append(record)
    else:
        kept.append(record)

# WRITE OUTPUT
SeqIO.write(kept, output_fasta, "fasta")

# REPORT
print(f"Total windows      : {len(kept) + len(removed)}")
print(f"Excluded windows   : {len(removed)}")
print(f"Retained candidates: {len(kept)}")
print(f"\nFiltered windows written to: {output_fasta}")


Loaded 323 exclusion sequences
Total windows      : 372
Excluded windows   : 320
Retained candidates: 52

Filtered windows written to: filtered_windows.fa
